# 03 Plan B VGGT (vggt_108, NO BA)

目标：

1. 从 `shared/images_512/` + `subset_108.txt` 准备 VGGT 输入
2. 检查依赖与环境
3. 跑 VGGT reconstruction **禁用 BA**
4. 检查 sparse 结果
5. 整理成标准 3DGS 输入结构

这本 notebook 只负责 **vggt_108** 初始化主线，**USE_BA=False**。

In [15]:
from pathlib import Path
import shutil
import subprocess
import importlib

CV_ROOT = Path('/home/bzhang512/CV_Project')
DATASET_ROOT = CV_ROOT / 'dataset'
THIRD_PARTY = CV_ROOT / 'third_party'

SCENE = 'Re10k-1'
SCENE = 'DL3DV-2'
SCENE = '405841'
SUBSET_NAME = 'subset_108.txt'
VGGT_VARIANT = 'vggt_108'
OVERWRITE = False

SCENE_ROOT = DATASET_ROOT / SCENE
SHARED_ROOT = SCENE_ROOT / 'part1' / 'shared'
IMAGES_512_DIR = SHARED_ROOT / 'images_512'
SUBSETS_DIR = SHARED_ROOT / 'subsets'
PLAN_B_ROOT = SCENE_ROOT / 'part1' / 'planB'
WORK_DIR = PLAN_B_ROOT / VGGT_VARIANT
VGGT_REPO = THIRD_PARTY / 'vggt'

print('WORK_DIR =', WORK_DIR)
print('VGGT_REPO =', VGGT_REPO)

WORK_DIR = /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_108
VGGT_REPO = /home/bzhang512/CV_Project/third_party/vggt


## 1. 依赖和环境检查

注意：某些模块需要先 `import torch`，再导入相关扩展。

In [16]:
import torch
print('torch =', torch.__version__)

for name in ['hydra', 'ninja', 'pycolmap', 'trimesh']:
    try:
        importlib.import_module(name)
        print('[OK]', name)
    except Exception as e:
        print('[FAIL]', name, e)

try:
    import lightglue
    print('[OK] lightglue')
except Exception as e:
    print('[FAIL] lightglue', e)

torch = 2.7.0+cu128
[OK] hydra
[OK] ninja
[OK] pycolmap
[OK] trimesh
[OK] lightglue


## 2. 准备 VGGT 输入 workspace（使用 subset_108）

In [17]:
for d in [WORK_DIR, WORK_DIR / 'inputs', WORK_DIR / 'raw_vggt', WORK_DIR / 'converted_colmap', WORK_DIR / 'eval', WORK_DIR / 'gs_scene']:
    d.mkdir(parents=True, exist_ok=True)

subset_file = SUBSETS_DIR / SUBSET_NAME
selected = [line.strip() for line in subset_file.read_text().splitlines() if line.strip()]
scene_input_dir = WORK_DIR / 'inputs' / 'scene' / 'images'
scene_input_dir.mkdir(parents=True, exist_ok=True)

for name in selected:
    src = IMAGES_512_DIR / name
    dst = scene_input_dir / name
    if dst.exists() and not OVERWRITE:
        continue
    shutil.copy2(src, dst)

print('prepared VGGT images:', len(selected))

prepared VGGT images: 108


## 3. VGGT reconstruction 参数（**USE_BA=False**）

In [18]:
# 关键：禁用 Bundle Adjustment
# 注意：demo_colmap.py 在 use_ba=False 分支里会强制走 feedforward 逻辑：
# - shared_camera = False
# - camera_type = 'PINHOLE'
VGGT_ARGS = {
    'use_ba': False,
    'fine_tracking': False,
    'query_frame_num': 4,
    'max_query_pts': 1024,
    'conf_thres_value': 1.0,
}
VGGT_ARGS


{'use_ba': False,
 'fine_tracking': False,
 'query_frame_num': 4,
 'max_query_pts': 1024,
 'conf_thres_value': 1.0}

## 4. 执行 VGGT reconstruction

In [19]:
# 正确调用 demo_colmap.py：
# 1) 不传 --use_ba => BA 关闭
# 2) 不传 --output_dir => demo_colmap.py 只支持写到 scene_dir/sparse
# 3) 在无 BA 分支下，不要强行传 shared_camera / SIMPLE_PINHOLE 作为主假设
scene_dir = WORK_DIR / 'inputs' / 'scene'
cmd = [
    'python', 'demo_colmap.py',
    '--scene_dir', str(scene_dir),
    '--query_frame_num', '4',
    '--max_query_pts', '1024',
    '--conf_thres_value', '1.0',
]
print('Run this command from VGGT_REPO; BA is disabled because --use_ba is omitted:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=VGGT_REPO, check=True)


Run this command from VGGT_REPO; BA is disabled because --use_ba is omitted:
python demo_colmap.py --scene_dir /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_108/inputs/scene --query_frame_num 4 --max_query_pts 1024 --conf_thres_value 1.0
Arguments: {'scene_dir': '/home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_108/inputs/scene', 'seed': 42, 'use_ba': False, 'max_reproj_error': 8.0, 'shared_camera': False, 'camera_type': 'SIMPLE_PINHOLE', 'vis_thresh': 0.2, 'query_frame_num': 4, 'max_query_pts': 1024, 'fine_tracking': False, 'conf_thres_value': 1.0}
Setting seed as: 42
Using device: cuda
Using dtype: torch.bfloat16
Model loaded
[after loading model] allocated=4.69 GB, reserved=4.71 GB
Loaded 108 images from /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_108/inputs/scene/images
[after loading images] allocated=5.95 GB, reserved=5.98 GB


/home/bzhang512/CV_Project/third_party/vggt/demo_colmap.py:81: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=dtype):


[after running VGGT aggregator] allocated=33.47 GB, reserved=42.07 GB
[after running VGGT camera head] allocated=33.47 GB, reserved=42.07 GB
[after converting pose encoding to extrinsic and intrinsic] allocated=33.47 GB, reserved=42.07 GB
[after running VGGT depth head] allocated=33.69 GB, reserved=34.62 GB
[after processing VGGT outputs] allocated=33.47 GB, reserved=34.62 GB
[after VGGT] allocated=5.96 GB, reserved=34.62 GB
[before processing reconstruction BA or w/o BA] allocated=5.96 GB, reserved=34.62 GB
Converting to COLMAP format
[after processing reconstruction BA or w/o BA] allocated=5.96 GB, reserved=34.62 GB
Saving reconstruction to /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_108/inputs/scene/sparse


CompletedProcess(args=['python', 'demo_colmap.py', '--scene_dir', '/home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_108/inputs/scene', '--query_frame_num', '4', '--max_query_pts', '1024', '--conf_thres_value', '1.0'], returncode=0)

## 5. Sparse output 检查

In [20]:
for p in [WORK_DIR / 'raw_vggt', WORK_DIR / 'converted_colmap', WORK_DIR / 'gs_scene']:
    print('\n', p)
    if p.exists():
        for child in sorted(p.iterdir())[:20]:
            print('  ', child.name)
    else:
        print('  missing')


 /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_108/raw_vggt

 /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_108/converted_colmap

 /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_108/gs_scene


## 6. 整理成标准 3DGS scene

In [ ]:
# 把 VGGT 输出整理成 3DGS / Scaffold 可直接读取的 gs_scene/sparse/0 结构
import shutil

scene_dir = WORK_DIR / 'inputs' / 'scene'
images_src = scene_dir / 'images'
raw_sparse = scene_dir / 'sparse'
gs_scene = WORK_DIR / 'gs_scene'

if gs_scene.exists():
    shutil.rmtree(gs_scene)
(gs_scene / 'images').mkdir(parents=True, exist_ok=True)
(gs_scene / 'sparse' / '0').mkdir(parents=True, exist_ok=True)

for p in images_src.iterdir():
    dst = gs_scene / 'images' / p.name
    shutil.copy2(p, dst)

if not raw_sparse.exists():
    raise FileNotFoundError(f'Expected VGGT sparse output at {raw_sparse}, but it does not exist')

# demo_colmap.py 当前实际输出是扁平 sparse/，这里统一搬到 sparse/0
for p in raw_sparse.iterdir():
    if p.is_file():
        dst = gs_scene / 'sparse' / '0' / p.name
        shutil.copy2(p, dst)

print('Plan B gs_scene prepared at', gs_scene)
print('Expected 3DGS sparse/0 exists =', (gs_scene / 'sparse' / '0').exists())
print('sparse/0 contents =', sorted([p.name for p in (gs_scene / 'sparse' / '0').iterdir()]))


Plan B gs_scene prepared at /home/bzhang512/CV_Project/dataset/405841/part1/planB/vggt_108/gs_scene
Expected 3DGS sparse/0 exists = True
sparse/0 contents = ['cameras.bin', 'images.bin', 'points.ply', 'points3D.bin']


: 